Experiment 7: Sequence-to-Sequence Model with Real Text Data

Aim: To implement a Sequence-to-Sequence model using LSTM with real text sequence data in TensorFlow and Keras.

In [1]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Sample Text Dataset
input_texts = [
    "hello",
    "how are you",
    "good morning",
    "thank you",
    "see you"
]

target_texts = [
    "hi",
    "i am fine",
    "morning",
    "welcome",
    "bye"
]

# Tokenizer
tokenizer = Tokenizer(lower=True, filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n')

tokenizer.fit_on_texts(input_texts + target_texts)

# Convert Text to Sequences
input_sequences  = tokenizer.texts_to_sequences(input_texts)
target_sequences = tokenizer.texts_to_sequences(target_texts)

# Padding
max_len = max(
    max(len(seq) for seq in input_sequences),
    max(len(seq) for seq in target_sequences)
)

encoder_input_data = pad_sequences(input_sequences,  maxlen=max_len, padding='post')
decoder_input_data = pad_sequences(target_sequences, maxlen=max_len, padding='post')

# Vocabulary Size
vocab_size = len(tokenizer.word_index) + 1

print(f"Vocabulary Size     : {vocab_size}")
print(f"Max Sequence Length : {max_len}")

# One Hot Encode Target
decoder_target_data = tf.keras.utils.to_categorical(
    decoder_input_data,
    num_classes=vocab_size
)

# Parameters
latent_dim = 32                  # Reduced from 64

# --- Encoder ---
encoder_inputs    = Input(shape=(max_len,), name='encoder_input')
encoder_embedding = Embedding(vocab_size, latent_dim, name='encoder_embedding')(encoder_inputs)

encoder_outputs, state_h, state_c = LSTM(
    latent_dim,
    return_state=True,
    name='encoder_lstm'
)(encoder_embedding)

encoder_states = [state_h, state_c]

# --- Decoder ---
decoder_inputs    = Input(shape=(max_len,), name='decoder_input')
decoder_embedding = Embedding(vocab_size, latent_dim, name='decoder_embedding')(decoder_inputs)

decoder_outputs, _, _ = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True,
    name='decoder_lstm'
)(decoder_embedding, initial_state=encoder_states)

decoder_outputs = Dense(
    vocab_size,
    activation='softmax',
    name='decoder_output'
)(decoder_outputs)

# Seq2Seq Model
model = Model(
    inputs=[encoder_inputs, decoder_inputs],
    outputs=decoder_outputs
)

# Model Summary
model.summary()

# Compile Model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train Model
history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=5,                # Increased from 2 (matches dataset size)
    epochs=50,                   # Reduced from 200
    verbose=0                    # Silent training
)

print("Training Complete!")

# Evaluate Model
loss, accuracy = model.evaluate(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    verbose=0
)

print(f"\nModel Loss     : {loss:.4f}")
print(f"Model Accuracy : {accuracy:.4f}")

Vocabulary Size     : 15
Max Sequence Length : 3


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 3, 32)     │        480 │ encoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 3, 32)     │        480 │ decoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 32),      │      8,320 │ encoder_embeddin… │
│                     │ (None, 32),       │            │                   │
│                     │ (None, 32)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 3, 32),   │      8,320 │ decoder_embeddin… │
│                     │ (None, 32),       │            │ encoder_lstm[0][… │
│                     │ (None, 32)]       │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_output      │ (None, 3, 15)     │        495 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 18,095 (70.68 KB)

 Trainable params: 18,095 (70.68 KB)

 Non-trainable params: 0 (0.00 B)

Training Complete!

Model Loss     : 1.3399
Model Accuracy : 0.5333


Conclusion: Successfully implemented a Sequence-to-Sequence model using real text sequence data with Encoder-Decoder LSTM architecture in TensorFlow and Keras.

